In [1]:
import os

In [2]:
%pwd

'd:\\Coding\\Data science\\Smart-Inventory-Defect-Detection-Quality-Control-API\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Coding\\Data science\\Smart-Inventory-Defect-Detection-Quality-Control-API'

In [20]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelBuildingConfig:
    root_dir: Path
    in_channels: int
    out_channels: int
    features: list

In [21]:
from SIDD.utils.common import read_yaml, create_directories
from SIDD.constants import *

In [26]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_building_config(self) -> ModelBuildingConfig:
        config = self.config.model_building
        params = self.params.model_building

        model_building_config = ModelBuildingConfig(
            root_dir=config.root_dir,
            in_channels=config.in_channels,
            out_channels=config.out_channels,
            features=params.features
        )

        return model_building_config

In [27]:
from SIDD import *
import torch
import torch.nn as nn

In [32]:
class ModelBuilding:
    def __init__(self, config: ModelBuildingConfig):
        self.config = config

    def build_model(self):
        return UNet(self.config.in_channels, self.config.out_channels, self.config.features)

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)
        x = self.conv(x)
        return x

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels, features):
        super().__init__()
        self.inc = DoubleConv(in_channels, features[0])
        self.down1 = Down(features[0], features[1])
        self.down2 = Down(features[1], features[2])
        self.up1 = Up(features[2], features[1])
        self.up2 = Up(features[1], features[0])
        self.outc = OutConv(features[0], out_channels)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x = self.up1(x3, x2)
        x = self.up2(x, x1)
        logits = self.outc(x)
        return logits




In [33]:
try:
    config = ConfigurationManager()
    model_building_config = config.get_model_building_config()
    model_builder = ModelBuilding(config=model_building_config)
    model = model_builder.build_model()

    dummy = torch.randn(
        2,
        model_building_config.in_channels,
        256,
        1600
    )

    output = model(dummy)

    print(f"Input Shape : {dummy.shape}")
    print(f"Output Shape: {output.shape}")
    
except Exception as e:
    raise e

[2026-06-27 00:01:47,965: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 00:01:47,967: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 00:01:47,969: INFO: common: created directory at: artifacts]
Input Shape : torch.Size([2, 3, 256, 1600])
Output Shape: torch.Size([2, 4, 256, 1600])
